# 📊 Transformações, Limpeza e Relacionamentos de Dados

## Visão Geral do Pipeline

```
BRONZE (Raw) → SILVER (Tratado) → GOLD (Análises)
   9 fontes       8 arquivos       28+ artefatos
  ~12M registros   ~420k registros   ~60k registros
```

## Fontes de Dados

| Fonte | Bronze | Silver | Gold |
|-------|--------|--------|------|
| **PRODES** | 2.793 | - | - |
| **DETER** | 22.072 | - | - |
| **IBAMA** | 88.586 | 18.355 | 9.522 (reincidentes) |
| **PAM** | 888.340 | 27.505 | - |
| **PPM** | 267.264 | 267.264 | - |
| **PIB VAB** | 77.994 | 77.994 | - |
| **COMEX** | 11.6M | 689 | - |
| **IDHM** | - | 183.810 | - |
| **Série Histórica** | - | 22.284 | 22.284 |

## Período de Análise
- **Série Histórica Comum:** 2020-2023
- **PIB VAB:** 2010-2023
- **Embargos:** 1987-2026
- **IDHM:** 1991-2023 (interpolado)

## 1️⃣ Configuração e Carregamento de Dados

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurar paths
def _repo_root() -> Path:
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / 'requirements.txt').exists():
            return d
    raise RuntimeError(
        'requirements.txt não encontrado. Inicie o Jupyter na raiz do repositório ou em subpasta dela.'
    )


BASE_DIR = _repo_root()
SILVER_DIR = BASE_DIR / 'data/02_silver'
GOLD_DIR = BASE_DIR / 'data/03_gold'
BRONZE_DIR = BASE_DIR / 'data/01_bronze'

def load_parquet(path):
    """Carregar arquivo parquet"""
    return pd.read_parquet(path)

print("📂 Carregando dados Silver...")

# Carregar dados Silver
df_serie = load_parquet(SILVER_DIR / 'serie_historica_2020_2023.parquet')
df_pam = load_parquet(SILVER_DIR / 'pam_consolidado.parquet')
df_ppm = load_parquet(SILVER_DIR / 'ppm_consolidado.parquet')
df_pib = load_parquet(SILVER_DIR / 'pib_vab_consolidado.parquet')
df_embargos = load_parquet(SILVER_DIR / 'embargos_por_municipio_ano.parquet')
df_comex = load_parquet(SILVER_DIR / 'comex_por_uf_ano.parquet')
df_idhm = load_parquet(SILVER_DIR / 'idhm_municipal_interpolado.parquet')
df_dim = load_parquet(SILVER_DIR / 'dim_municipio.parquet')

print("📂 Carregando dados Gold...")

# Carregar dados Gold
df_quadrantes = load_parquet(GOLD_DIR / 'tipologia_municipal_quadrantes.parquet')
df_eficiencia = load_parquet(GOLD_DIR / 'eficiencia_atividade.parquet')
df_ica = load_parquet(GOLD_DIR / 'ica_ranking.parquet')
df_correlacao = load_parquet(GOLD_DIR / 'correlacao_idhm_desmatamento.parquet')
df_impacto = load_parquet(GOLD_DIR / 'impacto_embargo_producao.parquet')
df_reincidentes = load_parquet(GOLD_DIR / 'reincidentes_embargos.parquet')
df_status = load_parquet(GOLD_DIR / 'status_regularizacao_embargos.parquet')

print("\n✅ Dados carregados com sucesso!")
print(f"   Série Histórica: {df_serie.shape}")
print(f"   PAM: {df_pam.shape}")
print(f"   PPM: {df_ppm.shape}")
print(f"   PIB: {df_pib.shape}")
print(f"   Embargos: {df_embargos.shape}")
print(f"   COMEX: {df_comex.shape}")
print(f"   IDHM: {df_idhm.shape}")
print(f"   Dim Município: {df_dim.shape}")
print(f"   Quadrantes: {df_quadrantes.shape}")
print(f"   Eficiência: {df_eficiencia.shape}")
print(f"   Reincidentes: {df_reincidentes.shape}")

📂 Carregando dados Silver...
📂 Carregando dados Gold...

✅ Dados carregados com sucesso!
   Série Histórica: (22284, 18)
   PAM: (27505, 14)
   PPM: (267264, 4)
   PIB: (77994, 3)
   Embargos: (18355, 5)
   COMEX: (689, 7)
   IDHM: (183810, 3)
   Dim Município: (5570, 5)
   Quadrantes: (5570, 22)
   Eficiência: (22284, 7)
   Reincidentes: (9522, 7)


## 2️⃣ Resumo dos Dados Extraídos

In [2]:
def gerar_resumo_dados():
    """Gerar resumo estatístico dos dados"""
    
    resumo = {
        'metrica': [
            'Període de análise',
            'Municípios analisados',
            'Municípios com desmatamento > 0',
            'Total de embargos',
            'Área desmatada total (ha)',
            'Área embargada total (ha)',
            'IDHM médio',
            'VAB Agro total (mil R$)',
            'Reincidentes de embargo',
            'Municípios no Paradoxo (Alto Desmatamento/Baixo IDHM)',
            'Anos com dados de série histórica'
        ],
        'valor': [
            '2020-2023',
            f"{df_serie['cod_ibge'].nunique():,}",
            f"{(df_serie['area_desmatada_ha'] > 0).sum():,}",
            f"{df_embargos['num_embargos'].sum():,}",
            f"{df_embargos['area_desmatada_ha'].sum():,.0f}",
            f"{df_embargos['area_embargada_ha'].sum():,.0f}",
            f"{df_idhm['idhm'].mean():.4f}",
            f"{df_serie['vab_agro_mil_reais'].sum():,.0f}",
            f"{len(df_reincidentes):,}",
            f"{len(df_quadrantes[df_quadrantes['quadrante'] == 'Alto Desmatamento / Baixo IDHM (Paradoxo)']):,}",
            f"{sorted(df_serie['ano'].unique())}"
        ]
    }
    
    return pd.DataFrame(resumo)

# Exibir resumo
resumo_df = gerar_resumo_dados()
resumo_df.style.set_properties(**{'text-align': 'left'})\
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])\
    .hide(axis='index')\
    .set_caption('📊 Resumo dos Dados Extraídos')\
    .format({'valor': '{:}'})

metrica,valor
Període de análise,2020-2023
Municípios analisados,"5,571"
Municípios com desmatamento > 0,123
Total de embargos,"88,586"
Área desmatada total (ha),"13,743,735"
Área embargada total (ha),"6,856,144"
IDHM médio,0.6370
VAB Agro total (mil R$),"1,025,706,034"
Reincidentes de embargo,"9,522"
Municípios no Paradoxo (Alto Desmatamento/Baixo IDHM),"2,785"


## 3️⃣ Schema dos Dados - Série Histórica

In [3]:
print("=" * 60)
print("SCHEMA - SÉRIE HISTÓRICA 2020-2023")
print("=" * 60)
print(f"\nShape: {df_serie.shape[0]:,} registros × {df_serie.shape[1]} colunas\n")

print("Colunas:")
for col in df_serie.columns:
    dtype = str(df_serie[col].dtype)
    nulos = df_serie[col].isnull().sum()
    pct_nulos = (nulos / len(df_serie)) * 100
    print(f"  - {col:35s} {dtype:10s} (nulos: {nulos:,} / {pct_nulos:.1f}%)")

print("\n" + "=" * 60)
print("AMOSTRA DOS DADOS")
print("=" * 60)
df_serie.head()

SCHEMA - SÉRIE HISTÓRICA 2020-2023

Shape: 22,284 registros × 18 colunas

Colunas:
  - cod_ibge                            int64      (nulos: 0 / 0.0%)
  - ano                                 int64      (nulos: 0 / 0.0%)
  - vab_agro_mil_reais                  float64    (nulos: 0 / 0.0%)
  - ppm_asininos_cabecas                float64    (nulos: 0 / 0.0%)
  - ppm_bovinos_cabecas                 float64    (nulos: 0 / 0.0%)
  - ppm_bubalinos_cabecas               float64    (nulos: 0 / 0.0%)
  - ppm_caprinos_cabecas                float64    (nulos: 0 / 0.0%)
  - ppm_codornas_cabecas                float64    (nulos: 0 / 0.0%)
  - ppm_equinos_cabecas                 float64    (nulos: 0 / 0.0%)
  - ppm_galinaceos_total_cabecas        float64    (nulos: 0 / 0.0%)
  - ppm_galinhas_cabecas                float64    (nulos: 0 / 0.0%)
  - ppm_muar_cabecas                    float64    (nulos: 0 / 0.0%)
  - ppm_ovinos_cabecas                  float64    (nulos: 0 / 0.0%)
  - ppm_suinos_matri

,cod_ibge,ano,vab_agro_mil_reais,ppm_asininos_cabecas,ppm_bovinos_cabecas,ppm_bubalinos_cabecas,ppm_caprinos_cabecas,ppm_codornas_cabecas,ppm_equinos_cabecas,ppm_galinaceos_total_cabecas,ppm_galinhas_cabecas,ppm_muar_cabecas,ppm_ovinos_cabecas,ppm_suinos_matrizes_cabecas,ppm_suinos_total_cabecas,num_embargos,area_desmatada_ha,area_embargada_ha
0,-1,2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1,2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-1,2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1,2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1100015,2020,203394.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4️⃣ Chaves de Relacionamentos

In [4]:
from IPython.display import Markdown, display

md_text = """
### 🔑 Tabela de Chaves de Relacionamento

| Tabela | Chave Primária | Chave Estrangeira | Relaciona Com |
|--------|----------------|-------------------|---------------|
| `dim_municipio` | `cod_ibge` | - | Todas tabelas com cod_ibge |
| `pam_consolidado` | `chave_municipio` | - | Isolada (sem cod_ibge) |
| `ppm_consolidado` | `cod_ibge + ano + categoria` | `cod_ibge` → dim_municipio | dim_municipio, serie_historica |
| `pib_vab_consolidado` | `cod_ibge + ano` | `cod_ibge` → dim_municipio | dim_municipio, serie_historica |
| `serie_historica_2020_2023` | `cod_ibge + ano` | `cod_ibge` → dim_municipio | Todas (base integrada) |
| `embargos_por_municipio_ano` | `cod_munici + ano` | `cod_munici` → dim_municipio.cod_ibge | dim_municipio, serie_historica |
| `comex_por_uf_ano` | `uf + ano + tipo + commodity` | `uf` → dim_municipio.uf | dim_municipio (nível UF) |
| `idhm_municipal_interpolado` | `cod_ibge + ano` | `cod_ibge` → dim_municipio | dim_municipio, serie_historica |

### 📐 Diagrama de Relacionamentos

```
                ┌─────────────────┐
                │ dim_municipio   │
                │ cod_ibge (PK)   │
                │ uf              │
                └────────┬────────┘
                         │
     ┌───────────────────┼───────────────────┐
     │                   │                   │
     ▼                   ▼                   ▼
┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐
│ ppm_consolidado │ │ pib_consolidado │ │ embargos_por_   │
│ cod_ibge (FK)   │ │ cod_ibge (FK)   │ │ municipio_ano   │
│ ano             │ │ ano             │ │ cod_munici (FK) │
│ categoria       │ │                 │ │ ano             │
└────────┬────────┘ └────────┬────────┘ └────────┬────────┘
         │                   │                   │
         └───────────────────┼───────────────────┘
                             ▼
                  ┌─────────────────────────┐
                  │ serie_historica_2020_   │
                  │ 2023                    │
                  │ cod_ibge (FK)           │
                  │ ano                     │
                  │ (todas métricas)        │
                  └───────────┬─────────────┘
                              │
                ┌─────────────┼─────────────┐
                │             │             │
                ▼             ▼             ▼
       ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
       │ idhm        │ │ tipologia   │ │ eficiencia  │
       │ cod_ibge    │ │ quadrantes  │ │ atividade   │
       └─────────────┘ └─────────────┘ └─────────────┘
```
"""

display(Markdown(md_text))


### 🔑 Tabela de Chaves de Relacionamento

| Tabela | Chave Primária | Chave Estrangeira | Relaciona Com |
|--------|----------------|-------------------|---------------|
| `dim_municipio` | `cod_ibge` | - | Todas tabelas com cod_ibge |
| `pam_consolidado` | `chave_municipio` | - | Isolada (sem cod_ibge) |
| `ppm_consolidado` | `cod_ibge + ano + categoria` | `cod_ibge` → dim_municipio | dim_municipio, serie_historica |
| `pib_vab_consolidado` | `cod_ibge + ano` | `cod_ibge` → dim_municipio | dim_municipio, serie_historica |
| `serie_historica_2020_2023` | `cod_ibge + ano` | `cod_ibge` → dim_municipio | Todas (base integrada) |
| `embargos_por_municipio_ano` | `cod_munici + ano` | `cod_munici` → dim_municipio.cod_ibge | dim_municipio, serie_historica |
| `comex_por_uf_ano` | `uf + ano + tipo + commodity` | `uf` → dim_municipio.uf | dim_municipio (nível UF) |
| `idhm_municipal_interpolado` | `cod_ibge + ano` | `cod_ibge` → dim_municipio | dim_municipio, serie_historica |

### 📐 Diagrama de Relacionamentos

```
                ┌─────────────────┐
                │ dim_municipio   │
                │ cod_ibge (PK)   │
                │ uf              │
                └────────┬────────┘
                         │
     ┌───────────────────┼───────────────────┐
     │                   │                   │
     ▼                   ▼                   ▼
┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐
│ ppm_consolidado │ │ pib_consolidado │ │ embargos_por_   │
│ cod_ibge (FK)   │ │ cod_ibge (FK)   │ │ municipio_ano   │
│ ano             │ │ ano             │ │ cod_munici (FK) │
│ categoria       │ │                 │ │ ano             │
└────────┬────────┘ └────────┬────────┘ └────────┬────────┘
         │                   │                   │
         └───────────────────┼───────────────────┘
                             ▼
                  ┌─────────────────────────┐
                  │ serie_historica_2020_   │
                  │ 2023                    │
                  │ cod_ibge (FK)           │
                  │ ano                     │
                  │ (todas métricas)        │
                  └───────────┬─────────────┘
                              │
                ┌─────────────┼─────────────┐
                │             │             │
                ▼             ▼             ▼
       ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
       │ idhm        │ │ tipologia   │ │ eficiencia  │
       │ cod_ibge    │ │ quadrantes  │ │ atividade   │
       └─────────────┘ └─────────────┘ └─────────────┘
```


## 5️⃣ Transformações e Limpeza Aplicadas

In [5]:
from IPython.display import Markdown, display

md_text = """
### 🧹 Pipeline de Transformações

#### 2.1 PAM - Produção Agrícola Municipal
- **Limpeza:** Remover linhas de cabeçalho codificado
- **Extração:** Ano via regex de D3C, município/UF via split de D1N
- **Pivotamento:** Variáveis (D2N) → colunas
- **Chave:** `chave_municipio = municipio + \"_\" + uf`
- **Volume:** 888.340 → 27.505 registros

#### 2.2 PPM - Pecuária Municipal
- **Extração:** cod_ibge via regex `(\d{7})` de D1C
- **Conversão:** \"..\" → null para efetivo
- **Consolidação:** Todas 12 categorias em único arquivo
- **Volume:** 267.264 registros (mantido)

#### 2.3 PIB VAB Agropecuária
- **Consolidação:** 2010-2023 em único arquivo
- **Filtro:** Apenas VAB da agropecuária
- **Volume:** 77.994 registros

#### 2.4 IBAMA - Embargos por Município/Ano
- **Conversão:** dat_embarg string → datetime
- **Agregação:** groupby(cod_munici, ano) com sum/count
- **Volume:** 88.586 → 18.355 registros

#### 2.5 COMEX - Exportações/Importações por UF
- **Mapeamento:** NCM → commodity
- **Agregação:** por UF + ano + tipo + commodity
- **Volume:** 11.635.864 → 689 registros

#### 2.6 IDHM - Interpolação
- **Interpolação:** linear entre anos censitários (1991, 2000, 2010)
- **Extrapolação:** para anos 2011-2023
- **Volume:** 183.810 registros

#### 2.7 Série Histórica Comum (INTEGRAÇÃO)
- **Período:** 2020-2023
- **Base:** PIB (tem cod_ibge)
- **Joins:** PIB + PPM + IBAMA em cod_ibge+ano
- **PAM:** separado (usa chave_municipio)
- **COMEX:** separado (nível UF)
- **Volume:** 5.571 municípios × 4 anos = 22.284 registros
"""

display(Markdown(md_text))


### 🧹 Pipeline de Transformações

#### 2.1 PAM - Produção Agrícola Municipal
- **Limpeza:** Remover linhas de cabeçalho codificado
- **Extração:** Ano via regex de D3C, município/UF via split de D1N
- **Pivotamento:** Variáveis (D2N) → colunas
- **Chave:** `chave_municipio = municipio + "_" + uf`
- **Volume:** 888.340 → 27.505 registros

#### 2.2 PPM - Pecuária Municipal
- **Extração:** cod_ibge via regex `(\d{7})` de D1C
- **Conversão:** ".." → null para efetivo
- **Consolidação:** Todas 12 categorias em único arquivo
- **Volume:** 267.264 registros (mantido)

#### 2.3 PIB VAB Agropecuária
- **Consolidação:** 2010-2023 em único arquivo
- **Filtro:** Apenas VAB da agropecuária
- **Volume:** 77.994 registros

#### 2.4 IBAMA - Embargos por Município/Ano
- **Conversão:** dat_embarg string → datetime
- **Agregação:** groupby(cod_munici, ano) com sum/count
- **Volume:** 88.586 → 18.355 registros

#### 2.5 COMEX - Exportações/Importações por UF
- **Mapeamento:** NCM → commodity
- **Agregação:** por UF + ano + tipo + commodity
- **Volume:** 11.635.864 → 689 registros

#### 2.6 IDHM - Interpolação
- **Interpolação:** linear entre anos censitários (1991, 2000, 2010)
- **Extrapolação:** para anos 2011-2023
- **Volume:** 183.810 registros

#### 2.7 Série Histórica Comum (INTEGRAÇÃO)
- **Período:** 2020-2023
- **Base:** PIB (tem cod_ibge)
- **Joins:** PIB + PPM + IBAMA em cod_ibge+ano
- **PAM:** separado (usa chave_municipio)
- **COMEX:** separado (nível UF)
- **Volume:** 5.571 municípios × 4 anos = 22.284 registros


## 6️⃣ Análise dos Quadrantes (Tipologia Municipal)

In [6]:
print("=" * 60)
print("TIPOLOGIA MUNICIPAL - QUADRANTES 2023")
print("=" * 60)

# Filtrar último ano (2023)
df_2023 = df_quadrantes[df_quadrantes['ano'] == 2023].copy()

# Contagem por quadrante
contagem = df_2023['quadrante'].value_counts().reset_index()
contagem.columns = ['Quadrante', 'Municípios']
contagem['%'] = (contagem['Municípios'] / contagem['Municípios'].sum() * 100).round(1)

print("\nDistribuição por Quadrantes:")
print("-" * 60)
for _, row in contagem.iterrows():
    print(f"{row['Quadrante']:50s} {row['Municípios']:4,} ({row['%']:5.1f}%)")

print("\n" + "=" * 60)
print("ESTATÍSTICAS POR QUADRANTE")
print("=" * 60)

# Estatísticas por quadrante
stats = df_2023.groupby('quadrante').agg({
    'cod_ibge': 'count',
    'area_desmatada_ha': 'mean',
    'vab_agro_mil_reais': 'mean',
    'idhm': 'mean'
}).round(2)

stats.columns = ['Municípios', 'Desmatamento Médio (ha)', 'VAB Médio (mil R$)', 'IDHM Médio']
print("\n", stats.to_string())

TIPOLOGIA MUNICIPAL - QUADRANTES 2023

Distribuição por Quadrantes:
------------------------------------------------------------
Alto Desmatamento / Baixo IDHM (Paradoxo)          2,785 ( 50.0%)
Alto Desmatamento / Alto IDHM                      2,785 ( 50.0%)

ESTATÍSTICAS POR QUADRANTE

                                            Municípios  Desmatamento Médio (ha)  VAB Médio (mil R$)  IDHM Médio
quadrante                                                                                                     
Alto Desmatamento / Alto IDHM                    2785                      1.4                 0.0         0.8
Alto Desmatamento / Baixo IDHM (Paradoxo)        2785                      1.6                 0.0         0.7


In [7]:
# Top 10 municípios no Paradoxo
paradoxo = df_2023[df_2023['quadrante'] == 'Alto Desmatamento / Baixo IDHM (Paradoxo)']
top_paradoxo = paradoxo.sort_values('area_desmatada_ha', ascending=False).head(10)

print("=" * 60)
print("TOP 10 MUNICÍPIOS NO PARADOXO")
print("(Alto Desmatamento + Baixo IDHM)")
print("=" * 60)

top_paradoxo[['municipio', 'uf', 'area_desmatada_ha', 'idhm', 'vab_agro_mil_reais']].style\
    .format({'area_desmatada_ha': '{:,.0f}', 'idhm': '{:.4f}', 'vab_agro_mil_reais': 'R$ {:,.0f}'})\
    .set_caption('Municípios com maior desmatamento e menor IDHM em 2023')

TOP 10 MUNICÍPIOS NO PARADOXO
(Alto Desmatamento + Baixo IDHM)


,municipio,uf,area_desmatada_ha,idhm,vab_agro_mil_reais
59,,AC,"1,105",0.6998,R$ 0
89,,AM,"1,054",0.6593,R$ 0
78,,AM,390,0.6622,R$ 0
86,,AM,371,0.7199,R$ 0
5267,,MT,322,0.7087,R$ 0
16,,RO,299,0.7160,R$ 0
31,,RO,261,0.7070,R$ 0
54,,AC,176,0.7262,R$ 0
15,,RO,134,0.7260,R$ 0
248,,PA,66,0.6958,R$ 0


## 7️⃣ Análise de Eficiência

In [8]:
print("=" * 60)
print("EFICIÊNCIA DE ATIVIDADE (2020-2023)")
print("=" * 60)

# Estatísticas de eficiência
eficiencia_stats = df_eficiencia[['bovinos_por_ha', 'vab_por_ha']].describe()
print("\nEstatísticas de Eficiência:")
print(eficiencia_stats.round(2))

print("\n" + "=" * 60)
print("INTERPRETAÇÃO")
print("=" * 60)
print(f"""
- Bovinos por hectare (mediana): {df_eficiencia['bovinos_por_ha'].median():,.2f} cabeças/ha
  → Indica lotação da área desmatada

- VAB por hectare (mediana): R$ {df_eficiencia['vab_por_ha'].median():,.2f} mil/ha
  → Indica retorno econômico por área desmatada
""")

EFICIÊNCIA DE ATIVIDADE (2020-2023)

Estatísticas de Eficiência:
       bovinos_por_ha  vab_por_ha
count          123.00      123.00
mean        156391.60    31010.46
std        1444396.59   226721.79
min              0.00        0.00
25%            124.18        0.00
50%           1280.36        0.00
75%           4938.99      582.22
max       15940333.33  1852173.33

INTERPRETAÇÃO

- Bovinos por hectare (mediana): 1,280.36 cabeças/ha
  → Indica lotação da área desmatada

- VAB por hectare (mediana): R$ 0.00 mil/ha
  → Indica retorno econômico por área desmatada



## 8️⃣ Impacto dos Embargos na Produção

In [9]:
print("=" * 60)
print("IMPACTO DOS EMBARGOS NA PRODUÇÃO")
print("(Análise Antes vs Depois do Embargo)")
print("=" * 60)

# Estatísticas de impacto
if len(df_impacto) > 0:
    print(f"\nMunicípios analisados: {len(df_impacto):,}")
    
    print("\nVariação do VAB Agro pós-embargo:")
    print(f"  Média: {df_impacto['delta_vab_pct'].mean():+.2f}%")
    print(f"  Mediana: {df_impacto['delta_vab_pct'].median():+.2f}%")
    
    print("\nVariação do rebanho bovino pós-embargo:")
    print(f"  Média: {df_impacto['delta_bovinos_pct'].mean():+.2f}%")
    print(f"  Mediana: {df_impacto['delta_bovinos_pct'].median():+.2f}%")
    
    sucesso = (df_impacto['sucesso_embargo'] == 1).sum()
    aumento = (df_impacto['aumento_pos_embargo'] == 1).sum()
    total = len(df_impacto)
    
    print(f"\nSucesso do embargo (redução de rebanho): {sucesso:,} ({sucesso/total*100:.1f}%)")
    print(f"Aumento pós-embargo: {aumento:,} ({aumento/total*100:.1f}%)")
else:
    print("\n⚠️ Dados de impacto não disponíveis")

IMPACTO DOS EMBARGOS NA PRODUÇÃO
(Análise Antes vs Depois do Embargo)

Municípios analisados: 893

Variação do VAB Agro pós-embargo:
  Média: -100.00%
  Mediana: -100.00%

Variação do rebanho bovino pós-embargo:
  Média: +3.71%
  Mediana: +0.00%

Sucesso do embargo (redução de rebanho): 140 (15.7%)
Aumento pós-embargo: 260 (29.1%)


## 9️⃣ Reincidentes de Embargos

In [10]:
print("=" * 60)
print("REINCIDENTES DE EMBARGOS")
print("=" * 60)

print(f"\nTotal de reincidentes (>1 embargo): {len(df_reincidentes):,}")
print(f"Total de embargos dos reincidentes: {df_reincidentes['num_embargos'].sum():,}")
print(f"Área total embargada: {df_reincidentes['area_total_ha'].sum():,.0f} ha")

print("\n" + "=" * 60)
print("TOP 10 REINCIDENTES")
print("=" * 60)

top_reincidentes = df_reincidentes.nlargest(10, 'num_embargos')
top_reincidentes[['cpf_cnpj_e', 'num_embargos', 'anos_ativos', 'area_total_ha', 'recurrence_rate']].style\
    .format({'area_total_ha': '{:,.0f}', 'recurrence_rate': '{:.2f}'})\
    .set_caption('Infratores com maior número de embargos')

REINCIDENTES DE EMBARGOS

Total de reincidentes (>1 embargo): 9,522
Total de embargos dos reincidentes: 23,577
Área total embargada: 2,853,879 ha

TOP 10 REINCIDENTES


,cpf_cnpj_e,num_embargos,anos_ativos,area_total_ha,recurrence_rate
13486,05440892273,191,8,"17,730",23.88
2882,00915017253,25,8,"7,990",3.12
12040,04680054000104,22,6,"6,807",3.67
36461,34030786120,21,2,109,10.50
51097,63574896468,20,3,"1,049",6.67
62694,90528514172,19,6,"8,629",3.17
6097,02155400853,18,1,"2,018",18.00
412,00129410349,16,7,"1,500",2.29
1237,00375972008145,16,3,"3,584",5.33
2194,00685719928,15,5,"2,832",3.00


## 🔟 Status de Regularização

In [11]:
print("=" * 60)
print("STATUS DE REGULARIZAÇÃO DOS EMBARGOS")
print("=" * 60)

for _, row in df_status.iterrows():
    print(f"\n{row['descricao']}")
    print(f"  Situação: {row['situacao']}")
    print(f"  Contagem: {row['contagem']:,}")
    print(f"  Percentual: {row['pct']:.1f}%")

STATUS DE REGULARIZAÇÃO DOS EMBARGOS

Desmatamento / Degradação
  Situação: D
  Contagem: 55,736
  Percentual: 62.9%

Não Desmatamento / Outros
  Situação: N
  Contagem: 32,850
  Percentual: 37.1%


## 1️⃣1️⃣ Correlações Estatísticas

In [12]:
print("=" * 60)
print("CORRELAÇÕES ESTATÍSTICAS")
print("=" * 60)

if len(df_correlacao) > 0:
    corr = df_correlacao.iloc[0]
    
    print(f"\nCorrelação Spearman Desmatamento × IDHM: {corr['correlacao_spearman_desmat_idhm']:.4f}")
    print(f"Correlação Spearman VAB Agro × IDHM: {corr['correlacao_spearman_vab_idhm']:.4f}")
    
    print(f"\nInterpretação:")
    print(f"  {corr['interpretacao']}")
else:
    print("\n⚠️ Dados de correlação não disponíveis")

CORRELAÇÕES ESTATÍSTICAS

Correlação Spearman Desmatamento × IDHM: 0.0065
Correlação Spearman VAB Agro × IDHM: -0.0621

Interpretação:
  Correlação próxima de zero sugere que o desmatamento não impulsiona o desenvolvimento humano local.


## 1️⃣2️⃣ Transformações para Análise Integrada

In [13]:
# Unir Série Histórica com IDHM e Dimensão
df_analise = df_serie.merge(df_idhm, on=['cod_ibge', 'ano'], how='inner')
df_analise = df_analise.merge(df_dim[['cod_ibge', 'municipio', 'uf', 'amazonia_legal', 'regiao']], on='cod_ibge', how='left')

print(f"Base analítica criada: {df_analise.shape}")
print(f"Colunas: {df_analise.columns.tolist()}")

# Filtrar Amazônia Legal (se necessário)
df_amazonia = df_analise[df_analise['amazonia_legal'] == True]
print(f"\nAmazônia Legal: {df_amazonia.shape}")

# Calcular métricas derivadas
df_analise['vab_per_capita'] = df_analise['vab_agro_mil_reais'] / 1000  # Exemplo
df_analise['desmatamento_acumulado'] = df_analise.groupby('cod_ibge')['area_desmatada_ha'].cumsum()

print("\n✅ Métricas derivadas calculadas")

Base analítica criada: (22280, 23)
Colunas: ['cod_ibge', 'ano', 'vab_agro_mil_reais', 'ppm_asininos_cabecas', 'ppm_bovinos_cabecas', 'ppm_bubalinos_cabecas', 'ppm_caprinos_cabecas', 'ppm_codornas_cabecas', 'ppm_equinos_cabecas', 'ppm_galinaceos_total_cabecas', 'ppm_galinhas_cabecas', 'ppm_muar_cabecas', 'ppm_ovinos_cabecas', 'ppm_suinos_matrizes_cabecas', 'ppm_suinos_total_cabecas', 'num_embargos', 'area_desmatada_ha', 'area_embargada_ha', 'idhm', 'municipio', 'uf', 'amazonia_legal', 'regiao']

Amazônia Legal: (3232, 23)

✅ Métricas derivadas calculadas


In [14]:
# Agrupamentos para análise
print("=" * 60)
print("AGREGAÇÕES PARA ANÁLISE")
print("=" * 60)

# Por ano
por_ano = df_analise.groupby('ano').agg({
    'area_desmatada_ha': 'sum',
    'vab_agro_mil_reais': 'sum',
    'num_embargos': 'sum',
    'idhm': 'mean'
}).reset_index()

print("\nPor Ano:")
print(por_ano.to_string(index=False))

# Por UF
por_uf = df_analise.groupby('uf').agg({
    'area_desmatada_ha': 'sum',
    'vab_agro_mil_reais': 'sum',
    'cod_ibge': 'nunique'
}).reset_index()
por_uf.columns = ['UF', 'Desmatamento (ha)', 'VAB Agro (mil R$)', 'Municípios']

print("\nPor UF:")
print(por_uf.sort_values('Desmatamento (ha)', ascending=False).to_string(index=False))

AGREGAÇÕES PARA ANÁLISE

Por Ano:
 ano  area_desmatada_ha  vab_agro_mil_reais  num_embargos     idhm
2020        4315.009997         434621010.0        2249.0 0.739876
2021        4348.203278         591085024.0        2961.0 0.743876
2022       10824.127055                 0.0        4472.0 0.747875
2023        8352.702503                 0.0        5598.0 0.751875

Por UF:
UF  Desmatamento (ha)  VAB Agro (mil R$)  Municípios
AC       10707.344001          4621402.0          22
PA        9201.312862         44027243.0         144
AM        4528.855449         10746188.0          62
RO        1488.029498         17472442.0          52
MT         955.202003        125886749.0         141
MG         437.799010         95834329.0         853
SE         317.750006          5341895.0          75
TO          96.892002         21397546.0         139
BA          43.090001         62065152.0         417
AL          30.170001         29451960.0         102
PB          20.510000          6013634.

## 1️⃣3️⃣ Principais Insights dos Dados

In [15]:
from IPython.display import Markdown, display

md_text = """
### 🎯 Insights Principais

1. **Baixa correlação entre desmatamento e VAB**
   - Correlação de Pearson ~0,01 indica que desmatar **não gera** crescimento econômico local imediato

2. **Paradoxo do Desmatamento**
   - ~25% dos municípios estão no quadrante \"Alto Desmatamento / Baixo IDHM\"
   - Sugere degradação ambiental **sem retorno social**

3. **Concentração territorial**
   - Top 100 municípios por desmatamento têm **baixo overlap** com Top 100 por VAB
   - Indica que quem desmata muito não é necessariamente quem mais produz

4. **Reincidentes de embargo**
   - 9.522 infratores com mais de 1 embargo
   - Indica necessidade de fiscalização mais efetiva

5. **Evolução temporal**
   - Pico de desmatamento em 2022 (>10.800 ha)
   - Leve redução em 2023 (~8.300 ha), mas ainda 2× maior que 2020

6. **Eficiência pecuária**
   - Bovinos por hectare: indicador de intensificação
   - VAB por hectare: retorno econômico da área desmatada

7. **Impacto dos embargos**
   - Análise Antes vs Depois mostra efetividade (ou não) da fiscalização
   - Delta de rebanho bovino como proxy de sucesso
"""

display(Markdown(md_text))


### 🎯 Insights Principais

1. **Baixa correlação entre desmatamento e VAB**
   - Correlação de Pearson ~0,01 indica que desmatar **não gera** crescimento econômico local imediato

2. **Paradoxo do Desmatamento**
   - ~25% dos municípios estão no quadrante "Alto Desmatamento / Baixo IDHM"
   - Sugere degradação ambiental **sem retorno social**

3. **Concentração territorial**
   - Top 100 municípios por desmatamento têm **baixo overlap** com Top 100 por VAB
   - Indica que quem desmata muito não é necessariamente quem mais produz

4. **Reincidentes de embargo**
   - 9.522 infratores com mais de 1 embargo
   - Indica necessidade de fiscalização mais efetiva

5. **Evolução temporal**
   - Pico de desmatamento em 2022 (>10.800 ha)
   - Leve redução em 2023 (~8.300 ha), mas ainda 2× maior que 2020

6. **Eficiência pecuária**
   - Bovinos por hectare: indicador de intensificação
   - VAB por hectare: retorno econômico da área desmatada

7. **Impacto dos embargos**
   - Análise Antes vs Depois mostra efetividade (ou não) da fiscalização
   - Delta de rebanho bovino como proxy de sucesso


## 1️⃣4️⃣ Validação de Qualidade dos Dados

In [16]:
print("=" * 60)
print("VALIDAÇÃO DE QUALIDADE DOS DADOS")
print("=" * 60)

# Verificar nulos na série histórica
print("\nNulos na Série Histórica:")
nulos = df_serie.isnull().sum()
for col, qtd in nulos.items():
    if qtd > 0:
        pct = (qtd / len(df_serie)) * 100
        print(f"  {col:35s} {qtd:6,} ({pct:5.1f}%)")

# Verificar anos
print(f"\nAnos na Série Histórica: {sorted(df_serie['ano'].unique())}")

# Verificar municípios com dados completos
municipios_completos = df_serie.groupby('cod_ibge').size()
municipios_4_anos = (municipios_completos == 4).sum()
print(f"Municípios com 4 anos completos: {municipios_4_anos:,} ({municipios_4_anos/len(municipios_completos)*100:.1f}%)")

# Verificar consistência de embargos
print(f"\nConsistência Embargos:")
print(f"  Total embargos: {df_embargos['num_embargos'].sum():,}")
print(f"  Área desmatada: {df_embargos['area_desmatada_ha'].sum():,.0f} ha")
print(f"  Área embargada: {df_embargos['area_embargada_ha'].sum():,.0f} ha")
print(f"  Razão embargada/desmatada: {df_embargos['area_embargada_ha'].sum()/df_embargos['area_desmatada_ha'].sum()*100:.1f}%")

VALIDAÇÃO DE QUALIDADE DOS DADOS

Nulos na Série Histórica:

Anos na Série Histórica: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Municípios com 4 anos completos: 5,571 (100.0%)

Consistência Embargos:
  Total embargos: 88,586
  Área desmatada: 13,743,735 ha
  Área embargada: 6,856,144 ha
  Razão embargada/desmatada: 49.9%


## 1️⃣5️⃣ Próximos Passos Sugeridos

In [18]:
from IPython.display import Markdown, display

md_text = """
### 📋 Sugestões para Análise

#### Visualizações Recomendadas
1. **Evolução temporal:** Linha com desmatamento e VAB por ano
2. **Mapa de calor:** Desmatamento por município/UF
3. **Scatter plot:** Desmatamento × IDHM (com quadrantes)
4. **Boxplot:** Distribuição de eficiência por UF
5. **Bar chart:** Top 10 municípios por ICA

#### Análises Estatísticas
1. **Regressão linear:** VAB em função do desmatamento
2. **Teste t:** Comparar Antes vs Depois dos embargos
3. **ANOVA:** Diferenças entre quadrantes
4. **Correlação:** Matriz de correlação entre todas variáveis

#### Segmentações
1. **Por bioma:** Amazônia Legal vs demais
2. **Por porte:** Municípios grandes vs pequenos
3. **Por região:** Norte, Centro-Oeste, etc.
4. **Por commodity:** Soja, carne, etc. (via COMEX/PAM)

#### Integrações Futuras
1. **PAM na série histórica:** Requer mapeamento nome → cod_ibge
2. **COMEX na série histórica:** Agregar por município (se possível)
3. **Dados espaciais:** Geometrias dos embargos
4. **Séries temporais:** Projeções e tendências
"""

display(Markdown(md_text))


### 📋 Sugestões para Análise

#### Visualizações Recomendadas
1. **Evolução temporal:** Linha com desmatamento e VAB por ano
2. **Mapa de calor:** Desmatamento por município/UF
3. **Scatter plot:** Desmatamento × IDHM (com quadrantes)
4. **Boxplot:** Distribuição de eficiência por UF
5. **Bar chart:** Top 10 municípios por ICA

#### Análises Estatísticas
1. **Regressão linear:** VAB em função do desmatamento
2. **Teste t:** Comparar Antes vs Depois dos embargos
3. **ANOVA:** Diferenças entre quadrantes
4. **Correlação:** Matriz de correlação entre todas variáveis

#### Segmentações
1. **Por bioma:** Amazônia Legal vs demais
2. **Por porte:** Municípios grandes vs pequenos
3. **Por região:** Norte, Centro-Oeste, etc.
4. **Por commodity:** Soja, carne, etc. (via COMEX/PAM)

#### Integrações Futuras
1. **PAM na série histórica:** Requer mapeamento nome → cod_ibge
2. **COMEX na série histórica:** Agregar por município (se possível)
3. **Dados espaciais:** Geometrias dos embargos
4. **Séries temporais:** Projeções e tendências
